In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd                                 
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score, 
)

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

x, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=42,
)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42
)

# Feature scaling is crucial for distance-based algorithms like KNN.
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# Cross-validation tuning for the optimal K.
k_values = [1, 3, 5, 7, 9, 11]
cv_score = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    score = cross_val_score(
        knn, x_train_scaled, y_train, cv=5, scoring="accuracy"
    )
    cv_score.append(score.mean())

best_k = k_values[np.argmax(cv_score)]
print(f"Optimal K selected by 5-fold cross-validation: K = {best_k}\n")

# Plot mean cross-validation accuracy for each K.
plt.figure(figsize=(8, 5))
plt.plot(k_values, cv_score, marker="o")
plt.title("KNN: Cross-Validation Accuracy vs. Number of Neighbors")
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Mean 5-fold CV Accuracy")
plt.xticks(k_values)
plt.ylim(0, 1)
plt.grid(True)
plt.show()


In [ ]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)

# sns.set_theme(style="whitegrid")

df = pd.read_csv(r"C:\Users\dipal\Downloads\Heart Disease Dataset.csv")

print("Dataset shape:", df.shape)
print()
print("Attributes:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

invalid_values = ["?", "NA", "N/A", "null", "None", ""]
df = df.replace(invalid_values, np.nan)

print("\nData after removing duplicates:", df.shape)

target_candidates = ["target", "Target", "HeartDisease", "heart_disease", "output", "condition"]

target_col = next(
    (column for column in target_candidates if column in df.columns),
    df.columns[-1]
)

print("\nTarget column:", target_col)

target_values = df[target_col].astype(str).str.strip().str.lower()

target_mapping = {
    "yes": 1,
    "no": 0,
    "presence": 1,
    "absence": 0,
    "positive": 1,
    "negative": 0,
    "true": 1,
    "false": 0,
    "disease": 1,
    "normal": 0
}

if target_values.isin(target_mapping).all():
    df[target_col] = target_values.map(target_mapping)
else:
    numeric_target = pd.to_numeric(df[target_col], errors="coerce")

    if numeric_target.notna().all():
        unique_values = sorted(numeric_target.unique())

        if len(unique_values) == 2:
            df[target_col] = numeric_target.map(
                {unique_values[0]: 0, unique_values[1]: 1}
            )
        else:
            df[target_col] = numeric_target
    else:
        df[target_col] = pd.factorize(target_values)[0]

df = df.dropna(subset=[target_col])

print("\nStatistical summary:")
print(df.describe(include=[np.number]))

print("\nTarget distribution:")
print(df[target_col].value_counts())

print("\nTarget proportions:")
print(df[target_col].value_counts(normalize=True))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col)
plt.title("Distribution of Heart Disease Target")
plt.xlabel("Heart Disease Class")
plt.ylabel("Number of Patients")
plt.show()

numeric_columns = df.drop(columns=[target_col]).select_dtypes(
    include=np.number
).columns.tolist()

if len(numeric_columns) < 2:
    raise ValueError("At least two numerical attributes are required.")

feature_one = "Age" if "Age" in df.columns else numeric_columns[0]
feature_two = "Chol" if "Chol" in df.columns else numeric_columns[1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(
    data=df,
    x=feature_one,
    hue=target_col,
    kde=True,
    element="step",
    ax=axes[0]
)
axes[0].set_title(f"Distribution of {feature_one}")

sns.boxplot(
    data=df,
    x=target_col,
    y=feature_two,
    ax=axes[1]
)
axes[1].set_title(f"{feature_two} by Heart Disease Class")

plt.tight_layout()
plt.show()

print(
    f"{feature_one} and {feature_two} were selected because they are commonly "
    "associated with cardiovascular risk. Differences in their distributions "
    "between the two target classes may indicate predictive relationships."
)

X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

# 8. Evaluate classification performance
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability)

print("\nModel Performance")
print("-----------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    cmap="Blues",
    values_format="d"
)
plt.title("Confusion Matrix")
plt.show()

false_positive_rate, true_positive_rate, _ = roc_curve(
    y_test,
    y_probability
)

plt.figure(figsize=(6, 4))
plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"Logistic Regression, AUC = {roc_auc:.3f}"
)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

unseen_predictions = X_test.copy()
unseen_predictions["Actual"] = y_test.values
unseen_predictions["Predicted"] = y_pred
unseen_predictions["Probability_of_Disease"] = y_probability

print("\nPredictions on unseen data:")
print(unseen_predictions.head(10))

print("\nSummary:")

print(
    f"The Logistic Regression model achieved an accuracy of {accuracy:.2f} and "
    f"a ROC-AUC score of {roc_auc:.2f} on the unseen test data."
)
print(
    "Precision, recall, F1-score, the confusion matrix, and ROC curve were used "
    "because accuracy alone may be misleading when the target classes are imbalanced."
)
print(
    "One limitation is that the model was evaluated using a single train-test split, "
    "so cross-validation and a larger independent dataset could provide more reliable results."
)